In [1]:
!pip install ipynb

In [ ]:
# main.ipynb
import tkinter as tk
from tkinter import messagebox, ttk
import re

from ipynb.fs.full.styles import (
    BACKGROUND_COLOR, CARD_COLOR, TEXT_COLOR, PRIMARY_COLOR, SECONDARY_COLOR,
    MUTED_TEXT, BORDER_COLOR, ERROR_COLOR, SUCCESS_COLOR,
    FONT_TITLE, FONT_SUBTITLE, FONT_BODY, FONT_SMALL, FONT_BUTTON, FONT_LABEL,
    BTN_PRIMARY, BTN_SECONDARY, BTN_DANGER,
    ENTRY_STYLE, LABEL_STYLE, FORM_LABEL_STYLE,
    SP_XS, SP_SM, SP_MD, SP_LG, SP_XL, SP_2XL, SP_3XL,
    SKILL_POOL
)

from ipynb.fs.full.data_manager import (
    initialise_file, save_user, validate_login,
    save_job, get_employer_jobs, get_job_applicants
)
from ipynb.fs.full.job_browsing import EmployeeDashboard

# ------------------------------------------------------------------ #
#  APP SETUP                                                           #
# ------------------------------------------------------------------ #

initialise_file()

root = tk.Tk()
root.title("WorkLink")
root.geometry("850x700")
root.config(bg=BACKGROUND_COLOR)
root.resizable(False, False)

# Session state
current_user_id   = None
current_user_name = None
current_user_role = None

# ------------------------------------------------------------------ #
#  SHARED HELPERS                                                      #
# ------------------------------------------------------------------ #

def clear_screen():
    """Destroys all widgets so a new screen can be built."""
    for widget in root.winfo_children():
        widget.destroy()


def logout():
    """Clears session and returns to the role-selection screen."""
    global current_user_id, current_user_name, current_user_role
    current_user_id   = None
    current_user_name = None
    current_user_role = None
    show_role_selection()


def make_scrollable_area(parent, inner_width=740):
    """Returns (canvas, scroll_frame) with a scrollbar already packed."""
    canvas = tk.Canvas(parent, bg=BACKGROUND_COLOR, highlightthickness=0)
    scrollbar = tk.Scrollbar(parent, orient="vertical", command=canvas.yview)
    scroll_frame = tk.Frame(canvas, bg=BACKGROUND_COLOR)

    scroll_frame.bind(
        "<Configure>",
        lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
    )
    canvas.create_window((0, 0), window=scroll_frame, anchor="nw", width=inner_width)
    canvas.configure(yscrollcommand=scrollbar.set)

    canvas.pack(side="left", fill="both", expand=True, padx=SP_LG)
    scrollbar.pack(side="right", fill="y")
    return canvas, scroll_frame


def make_form_field(parent, label_text, entry_type="entry", show=None, width=34):
    """
    Renders a consistently styled label + input pair.
    Returns the entry/text widget.
    """
    tk.Label(parent, text=label_text, **FORM_LABEL_STYLE).pack(
        anchor="w", padx=SP_3XL, pady=(SP_SM, SP_XS)
    )
    if entry_type == "text":
        widget = tk.Text(
            parent, width=width, height=5,
            font=FONT_BODY, bg=CARD_COLOR, fg=TEXT_COLOR,
            relief="solid", bd=1,
            highlightthickness=1,
            highlightcolor=PRIMARY_COLOR,
            highlightbackground=BORDER_COLOR
        )
    else:
        kwargs = {**ENTRY_STYLE, "width": width}
        if show:
            kwargs["show"] = show
        widget = tk.Entry(parent, **kwargs)
    widget.pack(pady=(0, SP_XS))
    return widget


def make_screen_header(parent, title_text, subtitle_text=None):
    """Renders a consistent page title + optional subtitle."""
    tk.Label(
        parent,
        text=title_text,
        font=FONT_TITLE,
        bg=BACKGROUND_COLOR,
        fg=PRIMARY_COLOR
    ).pack(pady=(SP_2XL, SP_XS))

    if subtitle_text:
        tk.Label(
            parent,
            text=subtitle_text,
            font=FONT_BODY,
            bg=BACKGROUND_COLOR,
            fg=MUTED_TEXT
        ).pack(pady=(0, SP_LG))


def make_link_button(parent, text, command):
    """Renders a subtle text-only link button."""
    return tk.Button(
        parent,
        text=text,
        font=("Helvetica", 10, "underline"),
        bg=BACKGROUND_COLOR,
        fg=PRIMARY_COLOR,
        bd=0,
        relief="flat",
        activebackground=BACKGROUND_COLOR,
        activeforeground=SECONDARY_COLOR,
        cursor="hand2",
        command=command
    )


# ------------------------------------------------------------------ #
#  SCREEN 1 — ROLE SELECTION                                          #
# ------------------------------------------------------------------ #

def show_role_selection():
    clear_screen()
    root.geometry("520x520")

    make_screen_header(
        root,
        "Welcome to WorkLink",
        "How would you like to get started?"
    )

    tk.Button(
        root, text="I'm an Employer",
        **BTN_PRIMARY, width=26, height=2,
        command=lambda: show_login("Employer")
    ).pack(pady=SP_MD)

    tk.Button(
        root, text="I'm a Job Seeker",
        **BTN_PRIMARY, width=26, height=2,
        command=lambda: show_login("Employee")
    ).pack(pady=SP_MD)

    tk.Button(
        root, text="Exit",
        **BTN_DANGER, width=14,
        command=root.destroy
    ).pack(pady=SP_2XL)


# ------------------------------------------------------------------ #
#  SCREEN 2 — LOGIN                                                   #
# ------------------------------------------------------------------ #

def show_login(role):
    clear_screen()
    root.geometry("520x520")

    role_label = "Employer" if role == "Employer" else "Job Seeker"
    make_screen_header(root, f"{role_label} Sign In")

    email_entry    = make_form_field(root, "Email")
    password_entry = make_form_field(root, "Password", show="*")

    def process_login():
        email = email_entry.get().strip()
        pwd   = password_entry.get().strip()

        if not email or not pwd:
            messagebox.showerror("Missing Information", "Please enter your email and password.")
            return

        profile = validate_login(email, pwd)
        if profile:
            if profile["role"] != role:
                messagebox.showerror(
                    "Wrong Account Type",
                    f"This account is registered as a {profile['role']}.\n"
                    f"Please go back and select the correct role."
                )
                return

            global current_user_id, current_user_name, current_user_role
            current_user_id   = str(profile["id"])
            current_user_name = profile["name"]
            current_user_role = profile["role"]

            if current_user_role == "Employer":
                show_employer_dashboard()
            else:
                show_employee_dashboard()
        else:
            messagebox.showerror("Sign In Failed", "Incorrect email or password. Please try again.")

    tk.Button(
        root, text="Sign In",
        **BTN_PRIMARY, width=22,
        command=process_login
    ).pack(pady=SP_XL)

    make_link_button(
        root,
        "New here? Create an account",
        lambda: show_register(role)
    ).pack(pady=SP_XS)

    tk.Button(
        root, text="← Back",
        **BTN_SECONDARY, width=18,
        command=logout
    ).pack(pady=SP_LG)


# ------------------------------------------------------------------ #
#  SCREEN 3 — REGISTER                                                #
# ------------------------------------------------------------------ #

def show_register(role):
    clear_screen()
    root.geometry("550x620")

    role_label = "Employer" if role == "Employer" else "Job Seeker"
    make_screen_header(root, f"Create {role_label} Account")

    # Name label differs per role so it feels personal
    name_label = "Company Name" if role == "Employer" else "Full Name"
    name_entry     = make_form_field(root, name_label)
    email_entry    = make_form_field(root, "Email")
    password_entry = make_form_field(root, "Password", show="*")

    skill_label = "Industry / Field" if role == "Employer" else "Primary Skill"
    tk.Label(root, text=skill_label, **FORM_LABEL_STYLE).pack(
        anchor="w", padx=SP_3XL, pady=(SP_SM, SP_XS)
    )
    selected_skill_var = tk.StringVar(value=SKILL_POOL[0])
    ttk.OptionMenu(root, selected_skill_var, SKILL_POOL[0], *SKILL_POOL).pack(pady=(0, SP_XS))

    def process_registration():
        name  = name_entry.get().strip()
        email = email_entry.get().strip()
        pwd   = password_entry.get().strip()
        skill = selected_skill_var.get()

        if not name or not email or not pwd:
            messagebox.showerror("Missing Information", "Please fill in all fields before continuing.")
            return

        if not re.match(r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$", email):
            messagebox.showerror("Invalid Email", "Please enter a valid email address.\nExample: name@example.com")
            return

        if len(pwd) < 6:
            messagebox.showerror("Password Too Short", "Your password must be at least 6 characters long.")
            return

        if len(name) > 50:
            messagebox.showerror("Name Too Long", "Name must be 50 characters or fewer.")
            return

        result = save_user(name, email, pwd, role, skill)

        if result == "LOCKED":
            messagebox.showerror(
                "Something went wrong",
                "We could not save your account right now. Please try again."
            )
        elif result:
            messagebox.showinfo(
                "Account Created!",
                "Your account has been set up. Please sign in to continue."
            )
            show_login(role)
        else:
            messagebox.showerror(
                "Email Already Registered",
                "An account with this email already exists. Try signing in instead."
            )

    tk.Button(
        root, text="Create Account",
        **BTN_PRIMARY, width=25,
        command=process_registration
    ).pack(pady=SP_XL)

    make_link_button(
        root,
        "Already have an account? Sign in",
        lambda: show_login(role)
    ).pack(pady=SP_XS)

    tk.Button(
        root, text="← Back",
        **BTN_SECONDARY, width=18,
        command=logout
    ).pack(pady=SP_LG)


# ------------------------------------------------------------------ #
#  SCREEN 4 — EMPLOYER DASHBOARD                                      #
# ------------------------------------------------------------------ #

def show_employer_dashboard():
    clear_screen()
    root.geometry("780x600")

    # Top bar: name + logout button in one row
    top_bar = tk.Frame(root, bg=CARD_COLOR, pady=SP_MD)
    top_bar.pack(fill="x")

    tk.Label(
        top_bar,
        text=f"👋  {current_user_name}",
        font=FONT_SUBTITLE,
        bg=CARD_COLOR,
        fg=TEXT_COLOR
    ).pack(side="left", padx=SP_XL)

    tk.Button(
        top_bar, text="Sign Out",
        **BTN_DANGER,
        command=logout
    ).pack(side="right", padx=SP_XL)

    # Page body
    body = tk.Frame(root, bg=BACKGROUND_COLOR)
    body.pack(fill="both", expand=True, padx=SP_3XL, pady=SP_2XL)

    make_screen_header(body, "Employer Dashboard", "What would you like to do?")

    tk.Button(
        body, text="Post a Job",
        **BTN_PRIMARY, width=30, height=2,
        command=show_post_job_screen
    ).pack(pady=SP_MD)

    tk.Button(
        body, text="View My Job Listings",
        **BTN_SECONDARY, width=30, height=2,
        command=show_employer_jobs_screen
    ).pack(pady=SP_MD)


# ------------------------------------------------------------------ #
#  SCREEN 5 — POST A JOB                                              #
# ------------------------------------------------------------------ #

def show_post_job_screen():
    clear_screen()
    root.geometry("780x680")

    make_screen_header(root, "Post a Job", "Fill in the details below to publish your listing.")

    # Job Title + counter
    tk.Label(root, text="Job Title", **FORM_LABEL_STYLE).pack(
        anchor="w", padx=SP_3XL, pady=(SP_SM, SP_XS)
    )
    title_var   = tk.StringVar()
    title_entry = tk.Entry(root, textvariable=title_var, width=50, **ENTRY_STYLE)
    title_entry.pack(pady=(0, SP_XS))

    title_counter = tk.Label(root, text="0 / 60", font=FONT_SMALL, bg=BACKGROUND_COLOR, fg=MUTED_TEXT)
    title_counter.pack(anchor="e", padx=SP_3XL)

    def on_title_change(*args):
        n = len(title_var.get())
        title_counter.config(
            text=f"{n} / 60",
            fg=ERROR_COLOR if n > 60 else MUTED_TEXT
        )
    title_var.trace_add("write", on_title_change)

    # Description + counter
    tk.Label(root, text="Job Description", **FORM_LABEL_STYLE).pack(
        anchor="w", padx=SP_3XL, pady=(SP_SM, SP_XS)
    )
    desc_entry = tk.Text(
        root, width=50, height=5,
        font=FONT_BODY, bg=CARD_COLOR, fg=TEXT_COLOR,
        relief="solid", bd=1,
        highlightthickness=1,
        highlightcolor=PRIMARY_COLOR,
        highlightbackground=BORDER_COLOR
    )
    desc_entry.pack(pady=(0, SP_XS))

    desc_counter = tk.Label(root, text="0 / 1000", font=FONT_SMALL, bg=BACKGROUND_COLOR, fg=MUTED_TEXT)
    desc_counter.pack(anchor="e", padx=SP_3XL)

    def on_desc_change(event):
        n = len(desc_entry.get("1.0", "end-1c"))
        desc_counter.config(
            text=f"{n} / 1000",
            fg=ERROR_COLOR if n > 1000 else MUTED_TEXT
        )
    desc_entry.bind("<KeyRelease>", on_desc_change)

    # Required Skill
    tk.Label(root, text="Required Skill", **FORM_LABEL_STYLE).pack(
        anchor="w", padx=SP_3XL, pady=(SP_SM, SP_XS)
    )
    job_skill_var = tk.StringVar(value=SKILL_POOL[0])
    ttk.OptionMenu(root, job_skill_var, SKILL_POOL[0], *SKILL_POOL).pack(pady=(0, SP_XS))

    # Experience Level
    tk.Label(root, text="Experience Level", **FORM_LABEL_STYLE).pack(
        anchor="w", padx=SP_3XL, pady=(SP_SM, SP_XS)
    )
    exp_var = tk.StringVar(value="Entry-Level")
    tk.OptionMenu(
        root, exp_var,
        "No Experience", "Entry-Level", "Intermediate", "Expert"
    ).pack(pady=(0, SP_SM))

    def handle_publish():
        t = title_var.get().strip()
        d = desc_entry.get("1.0", tk.END).strip()
        s = job_skill_var.get()
        e = exp_var.get()

        if not t or not d:
            messagebox.showerror("Missing Information", "Please fill in both the job title and description.")
            return
        if len(t) > 60:
            messagebox.showerror("Title Too Long", "Job title must be 60 characters or fewer.")
            return
        if len(d) > 1000:
            messagebox.showerror("Description Too Long", "Job description must be 1000 characters or fewer.")
            return

        result = save_job(current_user_id, t, d, s, e)

        if result == "LOCKED":
            messagebox.showerror(
                "Something went wrong",
                "Your job could not be saved right now. Please try again."
            )
        elif result:
            messagebox.showinfo("Job Posted!", "Your job listing is now live.")
            show_employer_dashboard()
        else:
            messagebox.showerror("Error", "Something went wrong. Please try again.")

    # Action buttons — row layout so they sit side by side
    btn_row = tk.Frame(root, bg=BACKGROUND_COLOR)
    btn_row.pack(pady=SP_XL)

    tk.Button(
        btn_row, text="Publish Job",
        **BTN_PRIMARY, width=20,
        command=handle_publish
    ).pack(side="left", padx=SP_SM)

    tk.Button(
        btn_row, text="Cancel",
        **BTN_SECONDARY, width=14,
        command=show_employer_dashboard
    ).pack(side="left", padx=SP_SM)


# ------------------------------------------------------------------ #
#  SCREEN 6 — EMPLOYER: VIEW JOBS                                     #
# ------------------------------------------------------------------ #

def show_employer_jobs_screen():
    clear_screen()
    root.geometry("780x650")

    # Sticky header row
    header_row = tk.Frame(root, bg=BACKGROUND_COLOR)
    header_row.pack(fill="x", padx=SP_2XL, pady=(SP_XL, SP_SM))

    tk.Label(
        header_row, text="My Job Listings",
        font=FONT_TITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR
    ).pack(side="left")

    tk.Button(
        header_row, text="← Dashboard",
        **BTN_SECONDARY,
        command=show_employer_dashboard
    ).pack(side="right")

    # Scrollable job cards
    _, scroll_frame = make_scrollable_area(root)

    my_jobs = get_employer_jobs(current_user_id)

    if not my_jobs:
        tk.Label(
            scroll_frame,
            text="You haven't posted any jobs yet.",
            font=FONT_SUBTITLE, bg=BACKGROUND_COLOR, fg=MUTED_TEXT
        ).pack(pady=SP_3XL)
        return

    for job in my_jobs:
        _render_employer_job_card(scroll_frame, job)


def _render_employer_job_card(parent, job):
    """Renders an employer-side job card with a button to view applicants."""
    card = tk.Frame(
        parent,
        bg=CARD_COLOR,
        highlightthickness=1,
        highlightbackground=BORDER_COLOR
    )
    card.pack(fill="x", padx=SP_LG, pady=SP_SM)

    inner = tk.Frame(card, bg=CARD_COLOR)
    inner.pack(fill="x", padx=SP_XL, pady=SP_MD)

    # Title
    tk.Label(
        inner, text=job["title"],
        font=FONT_SUBTITLE, bg=CARD_COLOR, fg=PRIMARY_COLOR, anchor="w"
    ).pack(fill="x")

    # Metadata row
    meta = tk.Frame(inner, bg=CARD_COLOR)
    meta.pack(fill="x", pady=(SP_XS, SP_SM))
    tk.Label(meta, text=f"🎯  {job['skills']}",      font=FONT_SMALL, bg=CARD_COLOR, fg=MUTED_TEXT).pack(side="left")
    tk.Label(meta, text="  ·  ",                       font=FONT_SMALL, bg=CARD_COLOR, fg=MUTED_TEXT).pack(side="left")
    tk.Label(meta, text=f"📊  {job['experience']}",   font=FONT_SMALL, bg=CARD_COLOR, fg=MUTED_TEXT).pack(side="left")

    # Description preview
    desc = job["description"]
    preview = desc[:140] + "..." if len(desc) > 140 else desc
    tk.Label(
        inner, text=preview,
        font=FONT_BODY, bg=CARD_COLOR, fg=TEXT_COLOR,
        wraplength=560, justify="left", anchor="w"
    ).pack(fill="x", pady=(0, SP_MD))

    # Action button
    tk.Button(
        inner, text="See Who Applied",
        **BTN_PRIMARY,
        command=lambda jid=job["id"], jtitle=job["title"]: show_applicants_screen(jid, jtitle)
    ).pack(anchor="e")


# ------------------------------------------------------------------ #
#  SCREEN 7 — EMPLOYER: VIEW APPLICANTS                               #
# ------------------------------------------------------------------ #

def show_applicants_screen(job_id, job_title):
    clear_screen()
    root.geometry("780x650")

    # Header row
    header_row = tk.Frame(root, bg=BACKGROUND_COLOR)
    header_row.pack(fill="x", padx=SP_2XL, pady=(SP_XL, SP_SM))

    tk.Label(
        header_row,
        text=f"Applicants for: {job_title}",
        font=FONT_TITLE, bg=BACKGROUND_COLOR, fg=PRIMARY_COLOR,
        wraplength=560, anchor="w", justify="left"
    ).pack(side="left")

    tk.Button(
        header_row, text="← My Jobs",
        **BTN_SECONDARY,
        command=show_employer_jobs_screen
    ).pack(side="right")

    candidates = get_job_applicants(job_id)

    # Scrollable area for applicant cards
    _, scroll_frame = make_scrollable_area(root)

    if not candidates:
        tk.Label(
            scroll_frame,
            text="No one has applied for this role yet.",
            font=FONT_SUBTITLE, bg=BACKGROUND_COLOR, fg=MUTED_TEXT
        ).pack(pady=SP_3XL)
        return

    tk.Label(
        scroll_frame,
        text=f"{len(candidates)} applicant(s)",
        font=FONT_BODY, bg=BACKGROUND_COLOR, fg=MUTED_TEXT
    ).pack(anchor="w", padx=SP_LG, pady=(SP_SM, SP_XS))

    for candidate in candidates:
        _render_applicant_card(scroll_frame, candidate)


def _render_applicant_card(parent, candidate):
    """Renders a single applicant card with clear name/contact/skill hierarchy."""
    card = tk.Frame(
        parent,
        bg=CARD_COLOR,
        highlightthickness=1,
        highlightbackground=BORDER_COLOR
    )
    card.pack(fill="x", padx=SP_LG, pady=SP_SM)

    inner = tk.Frame(card, bg=CARD_COLOR)
    inner.pack(fill="x", padx=SP_XL, pady=SP_MD)

    # Name — most important
    tk.Label(
        inner, text=candidate["name"],
        font=FONT_SUBTITLE, bg=CARD_COLOR, fg=TEXT_COLOR, anchor="w"
    ).pack(fill="x")

    # Contact + skill on one row
    meta = tk.Frame(inner, bg=CARD_COLOR)
    meta.pack(fill="x", pady=(SP_XS, 0))
    tk.Label(meta, text=f"✉  {candidate['email']}",  font=FONT_BODY, bg=CARD_COLOR, fg=MUTED_TEXT).pack(side="left")
    tk.Label(meta, text="   ·   ",                    font=FONT_BODY, bg=CARD_COLOR, fg=MUTED_TEXT).pack(side="left")
    tk.Label(meta, text=f"🎯  {candidate['skills']}", font=FONT_BODY, bg=CARD_COLOR, fg=MUTED_TEXT).pack(side="left")


# ------------------------------------------------------------------ #
#  SCREEN 8 — EMPLOYEE DASHBOARD                                      #
# ------------------------------------------------------------------ #

def show_employee_dashboard():
    clear_screen()
    root.geometry("850x680")

    session = {
        "id":   str(current_user_id),
        "name": current_user_name,
        "role": current_user_role
    }

    # Main dashboard frame
    frame = tk.Frame(root, bg=BACKGROUND_COLOR)
    frame.pack(fill="both", expand=True)

    panel = EmployeeDashboard(master=frame, current_user=session)
    panel.pack(fill="both", expand=True)

    # Logout button — compact, bottom-right corner, not a full banner
    logout_bar = tk.Frame(root, bg=BACKGROUND_COLOR)
    logout_bar.pack(fill="x", pady=SP_SM)

    tk.Button(
        logout_bar, text="Sign Out",
        **BTN_DANGER, width=14,
        command=logout
    ).pack(side="right", padx=SP_XL)


# ------------------------------------------------------------------ #
#  LAUNCH                                                             #
# ------------------------------------------------------------------ #

logout()          # Shows the role-selection screen on startup
root.mainloop()
